# Pipeline Medallion CITY (Bronze -> Silver -> Gold)

Ce notebook charge le CSV en Bronze (PySpark), puis construit Silver et Gold en Spark SQL avec nommage 4 parties `workspace.lakehouse.schema.table`.

In [ ]:
# Bronze: creation du schema CITY dans le Lakehouse par defaut (lkh_brz_demo)
spark.sql("CREATE SCHEMA IF NOT EXISTS CITY")

# Lecture du CSV bronze (header=true, inferSchema=true)
df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("Files/CITY/city_safety_seattle.csv")
)

# Conserver strictement les 11 colonnes de l'entete
if len(df.columns) != 11:
    raise ValueError(f"Le CSV attendu doit contenir 11 colonnes, trouve: {len(df.columns)}")

df = df.select(*df.columns)

# Ecriture idempotente en Bronze
(
    df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("CITY.city_safety")
)

print(f"Bronze CITY.city_safety row count: {df.count()}")

In [ ]:
%%sql
CREATE SCHEMA IF NOT EXISTS ws_assisted_dev.lkh_slv_demo.CITY;

CREATE OR REPLACE TABLE ws_assisted_dev.lkh_slv_demo.CITY.city_safety AS
SELECT
  dataSubtype,
  category,
  NULLIF(TRIM(subcategory), 'NULL') AS subcategory,
  NULLIF(TRIM(status), 'NULL') AS status,
  NULLIF(TRIM(source), 'NULL') AS source,
  to_timestamp(dateTime, 'yyyy-MM-dd HH:mm:ss') AS event_datetime,
  CAST(latitude AS double) AS latitude,
  CAST(longitude AS double) AS longitude,
  year(to_timestamp(dateTime, 'yyyy-MM-dd HH:mm:ss')) AS event_year,
  month(to_timestamp(dateTime, 'yyyy-MM-dd HH:mm:ss')) AS event_month,
  current_timestamp() AS Sid_LoadTimestamp,
  'city_safety_seattle.csv' AS Sid_SourceFile
FROM CITY.city_safety;

SELECT COUNT(*) AS silver_row_count
FROM ws_assisted_dev.lkh_slv_demo.CITY.city_safety;

In [ ]:
%%sql
CREATE SCHEMA IF NOT EXISTS ws_assisted_dev.lkh_gld_demo.CITY;

CREATE OR REPLACE TABLE ws_assisted_dev.lkh_gld_demo.CITY.city_safety_by_category AS
SELECT
  dataSubtype,
  category,
  event_year,
  event_month,
  COUNT(*) AS nb_incidents
FROM ws_assisted_dev.lkh_slv_demo.CITY.city_safety
GROUP BY
  dataSubtype,
  category,
  event_year,
  event_month;

SELECT COUNT(*) AS gold_row_count
FROM ws_assisted_dev.lkh_gld_demo.CITY.city_safety_by_category;